# ICT 5143 · Problem 9 — Fairness Preservation under Computational Budgets in Probabilistic Decision Systems
**Project notebook — runs on the REAL UCI Bank Marketing data (`bank-additional-full.csv`).** It drives the repository code in `src/` and `scripts/` (nothing is re-implemented here) and covers every item of the Problem 9 brief:

| Brief item | Where |
|---|---|
| Pre-contact decision time, leakage guard, provenance, small-group denominators | §1 |
| Fitted discrete BN, factorisation, diagram | §2 |
| **Exp 1** — VE (3 heuristics) = clique tree = enumeration within 1e-6 | §3 |
| Elimination traces, induced cliques, max clique size, evidence propagation | §4 |
| All experiments, statistics, figures (the `scripts/` pipeline) | §5 |
| **Exp 2** — preprocessing / query cost, peak factor size, memory by workload | §6 |
| **Exp 3** — operation budgets: approximation error, fallback rate, decisions by group | §7 |
| **Exp 4** — proposed vs equal-budget alternatives at matched utility; LR / GBT baselines; group-unaware routing | §8 |
| Paired block bootstrap, Holm correction | §9 |
| Required ablations and robustness tests | §10–11 |
| Secondary questions (large factors, clique-tree trade-off, errors near the threshold) | §12 |
| **Exp 5** — synthetic networks: treewidth × availability, with a no-group-dependence control | §13 |
| Wall-clock latency (kept separate from op budgets) | §14 |
| Post-hoc validation→test diagnostics, findings, requirement checklist | §15–16 |

**Safeguards enforced by the code:** `duration` and every post-contact field excluded · VE vs clique tree within 1e-6 · budgets are **operation counts**, never wall-clock · primary fairness metric **ΔEOpp** (age band × marital status), ΔDP and decision flips as diagnostics · groups with < 30 positives/negatives flagged **exploratory** and never silently dropped · synthetic masks/networks always labelled **SYNTHETIC**.

**How to run.** Keep the *updated* `afgml-p9.zip` next to this notebook (Colab: `/content`). Put `bank-additional-full.csv` (or the UCI `bank+marketing.zip`) in the same place, or let §1 download / ask you to upload it. The notebook **stops with an error if the real file is missing — it never falls back to synthetic data.** A full run takes tens of minutes the first time; results are cached in `results/` and reused while data, code and configs are unchanged.

In [ ]:
import sys, os, pathlib, subprocess, warnings, json, zipfile, shutil, hashlib, io, time, datetime
warnings.filterwarnings("ignore")

# ================================ SWITCHES ================================
RUN_PIPELINE = True     # run the experiment scripts when results/ is missing or stale
FORCE_RERUN  = False    # True = recompute everything even if cached results are current
OUT = "results"
# ==========================================================================

here = pathlib.Path.cwd().resolve()
def find_repo():
    for p in [here, *here.parents]:
        for c in (p, p / "afgml-p9"):
            if (c / "src" / "inference.py").exists(): return c
def find_zip():
    for d in dict.fromkeys([here, here.parent, pathlib.Path("/content"), pathlib.Path.home() / "Downloads"]):
        if d.exists():
            z = sorted(d.glob("afgml-p9*.zip"))
            if z: return z[0]

root = find_repo()
if root is None or not (root / "src" / "diagnostics.py").exists():     # missing repo, or an OLD copy of it
    z = find_zip()
    assert z, ("Upload the UPDATED afgml-p9.zip next to this notebook (Colab: /content), "
               "or open the notebook from inside the updated repository.")
    zipfile.ZipFile(z).extractall(z.parent)                            # overwrites code, keeps data/raw
    root = find_repo()
assert root is not None and (root / "src" / "diagnostics.py").exists(), "repo is missing src/diagnostics.py - use the updated zip"
os.chdir(root); sys.path.insert(0, str(root)); print("repo root:", root)

try:
    import numpy, pandas, sklearn, scipy, matplotlib, yaml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "pyyaml"])
import numpy as np, pandas as pd, yaml, matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display, Image
except ImportError:
    display, Image = print, (lambda **k: None)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)
D, M, C = (yaml.safe_load(open(f"configs/{n}.yaml")) for n in ("data", "model", "main"))
E = yaml.safe_load(open("configs/evaluation.yaml"))
PQ, H, MS, TOL = C["primary_budget_quantile"], C["heuristic"], C["min_group_support"], C["utility_loss_tol"]
bl = D["age_bands"]["labels"]
def show(df, nd=3): display(df.round(nd) if hasattr(df, "round") else df)
def fig(name):
    p = pathlib.Path(OUT) / "figures" / name
    if p.exists(): display(Image(filename=str(p)))
print(f"frozen config: capacity={C['capacity']}  primary budget quantile={PQ}  heuristic={H}  seeds={C['seeds']}  "
      f"AP-loss tolerance={TOL}  min group support={MS}")

## 1 · Data acquisition, provenance and the leakage guard
Decision time is **before the contact call**. `duration` (and anything observed during/after the call) is forbidden; `assert_no_leakage` enforces it. The cell below finds the real file (`data/raw/`, a UCI zip, a download, or a Colab upload) and **refuses to continue without it**.

In [ ]:
import io
NAME = "bank-additional-full.csv"
RAW = root / D["source_file"]; RAW.parent.mkdir(parents=True, exist_ok=True)

def _from_zip(zf):                                  # accepts bank+marketing.zip -> bank-additional.zip -> csv
    for n in zf.namelist():
        if n == NAME or n.endswith("/" + NAME): return zf.read(n)
    for n in zf.namelist():
        if n.lower().endswith(".zip"):
            got = _from_zip(zipfile.ZipFile(io.BytesIO(zf.read(n))))
            if got: return got

def _search_and_install():
    dirs = [root / "data" / "raw", root / "data", root, root.parent, here, pathlib.Path("/content"),
            pathlib.Path("/content/drive/MyDrive"), pathlib.Path.home() / "Downloads", pathlib.Path("/mnt/data")]
    for d in dict.fromkeys(dirs):
        if not d.exists(): continue
        for f in list(d.glob(NAME)) + [z for pat in ("bank*marketing*.zip", "bank-additional*.zip", "bank*.zip") for z in d.glob(pat)]:
            if f.resolve() == RAW.resolve(): continue
            blob = f.read_bytes() if f.suffix == ".csv" else _from_zip(zipfile.ZipFile(f))
            if blob: RAW.write_bytes(blob); print("found", f, "->", RAW); return True
    return False

if not RAW.exists(): _search_and_install()
if not RAW.exists():                                # try the project's own download script
    r = subprocess.run([sys.executable, "scripts/acquire_data.py"], capture_output=True, text=True, cwd=root)
    print((r.stdout + r.stderr)[-400:])
if not RAW.exists():                                # Colab: ask for an upload
    try:
        from google.colab import files
        print("Upload bank-additional-full.csv (or the UCI bank+marketing.zip)")
        for nm, blob in files.upload().items():
            (root / "data" / "raw" / nm).write_bytes(blob)
        _search_and_install()
    except ImportError:
        pass
if not RAW.exists():
    raise FileNotFoundError(
        f"Real data not found. Download {NAME} from https://archive.ics.uci.edu/dataset/222/bank+marketing "
        f"and place it at {RAW} (or put bank+marketing.zip next to this notebook), then re-run. "
        "This notebook never falls back to synthetic data.")

raw = pd.read_csv(RAW, sep=D["sep"])
EXPECTED = ["age", "job", "marital", "education", "default", "housing", "loan", "contact", "month", "day_of_week", "duration",
            "campaign", "pdays", "previous", "poutcome", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m",
            "nr.employed", "y"]
missing = [c for c in EXPECTED if c not in raw.columns]
assert not missing, f"not the bank-additional-full schema, missing {missing}"
assert set(raw["y"].unique()) <= {"yes", "no"}
prov = dict(file=str(RAW), sha256=hashlib.sha256(RAW.read_bytes()).hexdigest(), rows=int(len(raw)), cols=int(raw.shape[1]),
            n_yes=int((raw.y == "yes").sum()), accessed=str(datetime.date.today()), source="UCI Bank Marketing, dataset 222 (Moro, Rita, Cortez), CC BY 4.0")
prov["matches_published_uci_counts"] = (prov["rows"], prov["cols"], prov["n_yes"]) == (41188, 21, 4640)
print(json.dumps(prov, indent=1))
if not prov["matches_published_uci_counts"]:
    print("\n!!! WARNING: row/column/positive counts differ from the published file (41,188 x 21, 4,640 'yes'). "
          "Make sure this is the genuine UCI file before reporting anything. !!!")
os.makedirs(f"{OUT}/logs", exist_ok=True); json.dump(prov, open(f"{OUT}/logs/data_provenance.json", "w"), indent=1)
rd = root / "data" / "README.md"                                   # record SHA-256 / access date in the data card
if rd.exists() and "*(fill)*" in rd.read_text():
    rd.write_text(rd.read_text().replace("*(fill)*", f"{prov['accessed']} / `{prov['sha256']}`", 1))

In [ ]:
from src.data import prepare, assert_no_leakage, LeakageError, group_denominators
splits = prepare(raw, D)
print("model columns:", [c for c in splits["train"].columns if c not in ("y", "group", "marital_group")])
try:
    assert_no_leakage(D["bn_variables"] + ["duration"], D)
except LeakageError as e:
    print("LEAKAGE GUARD FIRED ->", e)
assert not any("duration" in c.lower() for f in splits.values() for c in f.columns)
print("OK: no post-contact column reaches any model. Macro/calendar columns excluded by declared choice:",
      not D["include_macro_columns"], not D["include_calendar_columns"])

print("\nPositional (date-ordered) 60/20/20 split — the release has no date column, so order = time:")
drift = pd.DataFrame({k: dict(n=len(v), n_positive=int(v.y.sum()), positive_rate=v.y.mean()) for k, v in splits.items()}).T
show(drift)
pr = drift.positive_rate
if pr.max() - pr.min() > 0.02:
    print(f"NOTE: the positive rate drifts across chronological splits (train {pr['train']:.3f}, val {pr['val']:.3f}, test {pr['test']:.3f}): "
          "settings chosen on validation are applied to a different base rate at test time. Reported, not hidden (see §15).")

print("\nGroup denominators (test split). 'exploratory' = fewer than 30 relevant observations; never silently dropped:")
gd = group_denominators(splits["test"], MS).sort_values("pos"); show(gd)

print("\nNATURAL evidence availability: share of 'unknown' (= unobserved) values by age band, test split")
mv = ["job", "education", "default", "housing", "loan"]
nat = pd.DataFrame({v: (splits["test"][v] == "unknown").groupby(splits["test"].age_band).mean() for v in mv})
nat["any_unknown"] = (splits["test"][mv] == "unknown").any(axis=1).groupby(splits["test"].age_band).mean()
show(nat)

## 2 · Bayesian network: factorisation, fitted tables, diagram
Nine discrete pre-contact variables plus `Y` (`configs/model.yaml`); CPDs are Dirichlet-smoothed counts fitted on **train only**. Marital status is used only to define audit groups, not as a model input. The evidence mask **R** is implemented as "variable unobserved ⇒ summed out", so *hidden evidence = more variables to eliminate = larger intermediate factors*.

In [ ]:
from src.bn import fit_bn, predict, evidence_matrix, row_evidence, verify_cgm_fitted
factors, states = fit_bn(splits["train"], M["structure"], M["alpha"], D)               # TRAIN only
print("P(Y | A, J, H, L, poutcome, pdays) · " + " · ".join(f"P({c} | {', '.join(p) or '∅'})" for c, p in M["structure"].items() if c != "y"))
print("\nvariables / cardinalities:", {v: len(s) for v, s in states.items()})
for f in factors:                                                                         # normalisation check (C-GM safeguard)
    assert np.allclose(f.values.sum(axis=-1), 1.0), f.vars
print("every CPD normalises over its child (checked)")
display(Image(filename="docs/bn_diagram.png"))

## 3 · Exp 1 — exact posteriors agree across elimination orders and clique-tree inference (tolerance 1e-6)
Elimination order changes **cost**, never the posterior. Checked three ways: (a) random tractable networks vs brute-force enumeration, (b) the fitted BN on real evidence patterns, (c) real patterns with extra synthetic masking.

In [ ]:
from src.inference import random_bn, variable_elimination, CliqueTree, brute_force, OpCounter
TOL_CGM = E["tolerance_cgm"]; rows = []
for seed in range(20):
    cpds, cards = random_bn(n=8, seed=seed); rng = np.random.default_rng(1000 + seed)
    cts = {h: CliqueTree(cpds, h) for h in ("min_fill", "min_weight", "min_degree")}
    for _ in range(10):
        names = list(cards); ev_v = rng.choice(names, size=int(rng.integers(0, 4)), replace=False).tolist()
        ev = {v: int(rng.integers(cards[v])) for v in ev_v}; q = [rng.choice([v for v in names if v not in ev])]
        ref, _ = brute_force(cpds, q, ev)
        for h, ct in cts.items():
            c = OpCounter(); p, _ = variable_elimination(cpds, q, ev, h, c)
            ct.calibrate(ev); pc, _ = ct.marginal(q)
            rows.append(dict(heuristic=h, err_ve=abs(p - ref).max(), err_ct=abs(pc - ref).max(), ve_ops=c.total, ve_peak=c.peak_factor_size))
t = pd.DataFrame(rows)
display(t.groupby("heuristic").agg(n=("err_ve", "size"), max_err_ve=("err_ve", "max"), max_err_ct=("err_ct", "max"),
                                  mean_ve_ops=("ve_ops", "mean"), mean_peak=("ve_peak", "mean")))
assert max(t.err_ve.max(), t.err_ct.max()) <= TOL_CGM
print(f"(a) random networks vs enumeration: PASS (tol {TOL_CGM:g})")

cg = verify_cgm_fitted(factors, states, splits["val"], n_patterns=40)
print("(b)/(c) fitted BN, real + masked evidence:", {k: (f"{v:.1e}" if isinstance(v, float) else v) for k, v in cg.items()})
assert cg["passed"]

## 4 · Graph analysis: induced cliques, elimination traces, evidence propagation
Maximum clique size / treewidth per heuristic (worst case, no evidence), then a **step-by-step trace** of the most expensive real test query under two heuristics (same posterior, different cost), and the calibrated clique tree with its two-pass sum-product message schedule.

In [ ]:
from src.diagnostics import (elimination_trace, induced_graph_summary, clique_tree_tables, evidence_pattern_costs, error_vs_threshold)
show(induced_graph_summary(factors))

ex_nat = predict(splits["test"], factors, states)                                   # exact, natural evidence
i_hard = ex_nat.ops.idxmax(); row = splits["test"].loc[i_hard]
ev = row_evidence(row, states)
print("most expensive natural test query: observed =", {v: states[v][s] for v, s in ev.items()},
      "| hidden =", [v for v in states if v != "y" and v not in ev])
for h in ("min_fill", "min_degree"):
    tr, post, ops = elimination_trace(factors, ["y"], ev, h)
    print(f"\nVE trace, heuristic={h}: total ops={ops}, P(y=1|evidence)={post[1]:.4f}, max induced clique width={tr.clique_width.max()}, "
          f"largest intermediate factor={tr.product_entries.max()} entries")
    show(tr[["step", "eliminated", "bucket_size", "induced_clique", "product_entries", "new_factor_scope", "ops_step", "ops_cum"]])

ct = CliqueTree(factors, "min_fill"); cl, ed, sch = clique_tree_tables(ct)
print("\nCalibrated clique tree (min_fill): cliques"); show(cl)
print("tree edges / separators"); show(ed)
print("sum-product message schedule (collect leaves→root, then distribute root→leaves)"); show(sch)
c = ct.calibrate(ev); pc, _ = ct.marginal(["y"])
print(f"\nevidence propagation: calibrating this evidence costs {c.total} ops (peak factor {c.peak_factor_size} entries); "
      f"P(y=1|evidence)={pc[1]:.4f} equals the VE posterior {post[1]:.4f} -> {np.isclose(pc[1], post[1])}")

## 5 · Run the full experiment pipeline on the real data
Runs the frozen scripts (`prepare_data` → `verify_inference` → `run_experiments` → `evaluate` → `make_results` → `profile_latency`). Settings are in `configs/*.yaml` and were **frozen before looking at test outcomes**: 5 seeds for every stochastic part, 1,000 block-bootstrap replicates, Holm correction for the confirmatory family. Output is cached; a changed dataset, source file or config triggers a re-run.

In [ ]:
def code_stamp():
    h = hashlib.sha256()
    for f in sorted((root / "src").glob("*.py")) + sorted((root / "scripts").glob("*.py")) + sorted((root / "configs").glob("*.yaml")):
        h.update(f.read_bytes())
    return h.hexdigest()
STAMP = dict(data=prov["sha256"], code=code_stamp())
NEED = ["metrics/main_results.csv", "metrics/bootstrap.csv", "metrics/seed_aggregate.csv", "metrics/exp2_complexity.csv",
        "metrics/exp5_aggregate.csv", "metrics/latency.csv", "metrics/cgm.json", "metrics/predictive_baselines.csv",
        "metrics/group_table.csv", "logs/run_log.json", "figures/fig_complexity.png"]
def results_current():
    sp = pathlib.Path(OUT) / "logs" / "notebook_stamp.json"
    return all((pathlib.Path(OUT) / n).exists() for n in NEED) and sp.exists() and json.load(open(sp)) == STAMP

def run(cmd):
    print("\n$ python", " ".join(cmd)); t0 = time.time()
    p = subprocess.Popen([sys.executable, "-u", *cmd], cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print("   ", line, end="")
    p.wait(); print(f"   [exit {p.returncode}, {time.time() - t0:.0f}s]")
    if p.returncode: raise RuntimeError(f"{cmd[0]} failed (exit {p.returncode}); see output above")

if results_current() and not FORCE_RERUN:
    print("cached results are current for this data + code + configs -> not re-running (set FORCE_RERUN=True to recompute)")
elif RUN_PIPELINE:
    for cmd in (["scripts/prepare_data.py", "configs/data.yaml"], ["scripts/verify_inference.py", "--mode", "real"],
                ["scripts/run_experiments.py", "--mode", "real", "--out", OUT],
                ["scripts/evaluate.py", "--out", OUT], ["scripts/make_results.py", "--out", OUT, "--mode", "real"],
                ["scripts/profile_latency.py", "--mode", "real", "--out", OUT]):
        run(cmd)
    json.dump(STAMP, open(f"{OUT}/logs/notebook_stamp.json", "w"))
else:
    assert results_current(), "RUN_PIPELINE=False but no current results/ found"

LOG = json.load(open(f"{OUT}/logs/run_log.json"))
assert LOG["mode"] == "real" and LOG["data_sha256"] == prov["sha256"], "results/ were not produced from this real dataset"
MET = lambda n: pd.read_csv(f"{OUT}/metrics/{n}.csv")
R, CH, GT = MET("main_results"), MET("chosen_fallbacks"), MET("group_table")
BUD = {float(k): v for k, v in LOG["budgets_ops"].items()}
print("run mode:", LOG["mode"], "| runtime (s):", LOG["runtime_s"], "| seeds:", LOG["seeds"])
print("deterministic op budgets (mult+add per query) at natural validation quantiles:", BUD)

## 6 · Exp 2 — preprocessing cost, query cost, peak factor size and memory across evidence workloads
Workloads: **natural** evidence (real `unknown` = unobserved) and **synthetic** extra masking at 25/50/75 % (*SYNTHETIC*). Costs are factor-entry operation counts (deterministic); memory is `tracemalloc` peak for one query.

In [ ]:
X2 = MET("exp2_complexity"); show(X2, 1)
fig("fig_complexity.png")
nat2 = X2[X2.workload == "natural"].set_index("method")
ve, ctm = nat2.loc[f"ve_{H}"], nat2.loc[f"clique_tree_{H}"]
print(f"Single-marginal query on natural evidence: VE({H}) mean {ve.ops_mean:.1f} ops, peak factor {ve.peak_factor_max:.0f} entries; "
      f"clique-tree calibration {ctm.ops_mean:,.0f} ops/evidence (+{ctm.preprocessing_ops:,.0f} one-off preprocessing), peak {ctm.peak_factor_max:.0f}, "
      f"max clique {ctm.max_clique:.0f} vars.\n"
      "-> Clique-tree calibration answers EVERY marginal for one evidence setting; for a single P(Y|e) query VE is far cheaper. "
      "The tree only pays off when many different marginals share the same evidence (secondary question 2).")
syn = X2[X2.method == f"ve_{H}"].set_index("workload")[["ops_mean", "ops_max", "peak_factor_mean", "peak_factor_max"]]
print("\nHow cost grows with hidden evidence (VE):"); show(syn, 1)

In [ ]:
print("Which evidence patterns create the LARGEST intermediate factors? (pattern = which variables are observed; test split, min_fill)")
from src.pipeline import make_mask
top_nat, all_nat = evidence_pattern_costs(factors, states, splits["test"], None, H, top=8)
print("\nNATURAL evidence — top patterns by peak factor size"); show(top_nat)
mt_dep = make_mask(splits["test"], C, bl, "dependent", 0, 1)
top_dep, all_dep = evidence_pattern_costs(factors, states, splits["test"], mt_dep, H, top=8)
print("\nSYNTHETIC dependent mask (young customers hide more variables) — top patterns by peak factor size"); show(top_dep)
corr = all_nat[["n_observed", "ops"]].corr().iloc[0, 1]
print(f"\ncorrelation(#observed variables, ops) over natural patterns = {corr:+.2f}: "
      "cost depends on WHICH evidence is hidden, and that is what an operation budget trips on.")

## 7 · Exp 3 — controlled operation budgets: approximation error, fallback rate and decisions by group
A query **trips** iff exact elimination would exceed the budget (counted in factor-entry multiplications + additions); a precomputed fallback table then answers it. Budgets are quantiles of the *natural* validation op-count distribution. `natural` = real missingness only; `dependent` = **SYNTHETIC** extra masking, heavier for the youngest band; `independent` = **SYNTHETIC** matched control with the same overall masking rate but no group dependence.

In [ ]:
MCOLS = ["ap", "auroc", "balanced_acc", "dEOpp_supported", "dEOpp", "dDP", "dEOdds", "sel_ratio", "fallback_rate", "fallback_gap",
         "worst_group_flip", "agree_exact", "tv_mean", "tv_max", "log_loss", "brier"]
RM = R[(R.tag == "main") & np.isclose(R.q, PQ)]
def agg(df, cols=MCOLS, by=("workload", "method")):
    g = df.groupby(list(by))[cols]
    out = g.mean().add_suffix("")
    out["n_seeds"] = df.groupby(list(by)).seed.nunique()
    return out
A = agg(RM)
core = ["exact", "global_prior", "proposed", "proposed_unaware", "util_sel", "thr_adj_post_budget", "fair_exact"]
nd = sorted(set(BUD.values()))
print(f"Budget grid (natural validation op-count quantiles -> ops): {BUD}  | {len(nd)} distinct budgets out of {len(BUD)} quantiles "
      "(op counts are small integers, so neighbouring quantiles can coincide — duplicates give identical results by construction).")
print(f"Primary budget q={PQ} -> {BUD[PQ]} ops. Mean over seeds (natural workload has no stochastic part -> 1 run).")
for w in ("natural", "dependent", "independent"):
    print(f"\n=== workload: {w}" + ("" if w == "natural" else "  (SYNTHETIC masks)"))
    show(A.loc[w].loc[[m for m in core if m in A.loc[w].index], ["n_seeds", "fallback_rate", "fallback_gap", "worst_group_flip", "agree_exact",
                                                                "tv_mean", "tv_max", "log_loss", "brier"]])

In [ ]:
print("Fallback rate and decision change vs exact, by age×marital group (primary budget, seed 0). Red/flagged = exploratory (<30 positives).")
g = GT[GT.method.isin(["global_prior", "proposed"])].copy()
g["exploratory"] = g.pos < MS
for w in ("natural", "dependent"):
    x = g[g.workload == w].pivot_table(index=["group", "n", "pos", "exploratory"], columns="method", values=["fallback_rate", "flip", "tpr"])
    x.columns = [f"{a}:{b}" for a, b in x.columns]
    print(f"\n--- {w}" + ("  (SYNTHETIC masks)" if w != "natural" else "")); show(x.sort_index(level="n", ascending=False))
fig("fig_group_fallback.png")

In [ ]:
print("Probabilistic error vs the exact posterior (binary => total-variation distance = |p - p_exact|), seed 0, primary budget")
GD = group_denominators(splits["test"], MS).set_index("group")
rows, grows = [], []
for w in ("natural", "dependent"):
    Z = np.load(f"{OUT}/runs/artifacts_{w}.npz", allow_pickle=True); pe = Z["score_exact"]
    for m in ("global_prior", "proposed", "proposed_unaware", "util_sel"):
        err = np.abs(Z[f"score_{m}"] - pe)
        rows.append(dict(workload=w, method=m, max_marginal_error=err.max(), mean_TV=err.mean(),
                         mean_TV_on_fallback_queries=err[Z[f"fb_{m}"].astype(bool)].mean() if Z[f"fb_{m}"].any() else np.nan))
        s = pd.Series(err).groupby(Z["group"]).mean().rename("groupwise_mean_marginal_error")
        grows.append(s.reset_index().rename(columns={"index": "group"}).assign(workload=w, method=m))
show(pd.DataFrame(rows), 4)
ge = pd.concat(grows); ge["pos"] = ge["group"].map(GD.pos); ge["exploratory"] = ge.pos < MS
print("\ngroupwise mean marginal error (largest first; exploratory groups flagged)")
show(ge[ge.workload == "dependent"].pivot_table(index=["group", "pos", "exploratory"], columns="method", values="groupwise_mean_marginal_error").sort_values("global_prior", ascending=False), 4)
print("log loss and Brier (per method, above) are computed on the budgeted probabilities; AUROC / AP / balanced accuracy in §8.")

In [ ]:
print("Budget–accuracy–fairness curves (mean ± 95% t-interval over seeds; budgets = natural validation op quantiles)")
fig("fig_budget_curves_natural.png"); fig("fig_budget_curves_dependent.png")

## 8 · Exp 4 — proposed fallback vs equal-budget alternatives at matched utility
**Proposed rule (frozen).** Among candidate fallbacks with *validation* AP loss ≤ 0.01 pick the one with the smallest worst-group decision-flip rate vs exact; routing itself uses only evidence-pattern/op-count information. `proposed_unaware` is the same rule restricted to fallbacks that **never** condition on the protected attribute (the required group-unaware routing comparison). `util_sel` (utility-only selection), `global_prior` (one global fallback), `thr_adj_post_budget` (Hardt-style threshold adjustment after budgeted inference; **needs the group label at decision time**) and `fair_exact` (fairness-adjusted exact rule; also needs it) are the baselines/ablations.

In [ ]:
def matched_utility(w):
    a = A.loc[w].copy(); ap_ex = a.loc["exact", "ap"]
    a["ap_loss_vs_exact"] = ap_ex - a["ap"]; a["within_utility_tol"] = a.ap_loss_vs_exact <= TOL
    return a
for w in ("natural", "dependent"):
    a = matched_utility(w)
    print(f"\n=== {w}" + ("" if w == "natural" else "  (SYNTHETIC masks)") + f"  — primary budget {BUD[PQ]} ops, capacity {C['capacity']:.0%}")
    show(a.loc[[m for m in core if m in a.index], ["ap", "ap_loss_vs_exact", "within_utility_tol", "auroc", "balanced_acc", "dEOpp_supported", "dDP", "dEOdds",
                                                  "worst_group_flip", "agree_exact", "fallback_rate"]])
uses = R[(R.tag == "main") & np.isclose(R.q, PQ)].groupby("method").group_label_at_decision.max()
print("\nmethods that use the protected attribute / group label at decision time in at least one run:",
      sorted(uses[uses].index.tolist()))
print("\nWhich fallback did each rule pick? (primary budget)")
ch = CH[(CH.tag == "main") & np.isclose(CH.q, PQ)]
for col in ("proposed", "proposed_unaware", "util_sel"):
    print(col); show(ch.groupby("workload")[col].value_counts().unstack(fill_value=0))

In [ ]:
print("Predictive baselines on the SAME pre-contact variables (logistic regression, gradient-boosted trees) vs the exact BN; mean over seeds")
PB = MET("predictive_baselines")
show(PB.groupby("method")[["ap", "auroc", "balanced_acc", "recall_at_capacity", "dEOpp_supported", "dDP", "dEOdds", "agree_exact", "log_loss", "brier"]].mean())
print("(decisions use the same validation-fixed top-20% threshold rule; 'agree_exact' = agreement with the exact-BN decisions)")

## 9 · Statistical validation — paired block bootstrap and Holm correction
Resampling unit = **contiguous date-ordered blocks of the test set** (moving-block bootstrap, block length 50, 1,000 replicates). Customer IDs are absent, so **dependence from repeated customers cannot be controlled** — intervals describe *evaluation* uncertainty of fixed models only, and seeds measure algorithmic (mask) variability, not population sampling. The statistic is ΔEOpp over **supported** groups (≥ 30 positives in the full test sample, fixed once) — exactly the `dEOpp_supported` reported everywhere. The earlier notebook bootstrapped over all groups including tiny ones (n = 2), which dominated the interval; the all-groups version is kept below as a labelled sensitivity.

In [ ]:
B = MET("bootstrap")
print(f"test n={int(B.n_test.dropna().iloc[0])}, block length={int(B.block_len.iloc[0])}, replicates={int(B.n_boot.min())}, alpha=0.05")
cols = ["workload", "comparison", "estimate", "lo", "hi", "p_two_sided", "p_holm", "n_groups_used"]
print("\nCONFIRMATORY family (natural workload, dEOpp, Holm-corrected): proposed − comparator, negative = proposed fairer")
show(B[B.kind == "proposed_minus_comparator"].query("confirmatory")[cols])
print("\nEXPLORATORY: dependent workload (SYNTHETIC masks), same comparisons (not in the confirmatory family)")
show(B[(B.kind == "proposed_minus_comparator") & (B.workload == "dependent")][cols])
print("\nEXPLORATORY: group-unaware routing")
show(B[B.kind == "exploratory_unaware"][cols])
print("\nDegradation of each method vs exact (ΔEOpp_method − ΔEOpp_exact)")
show(B[B.kind == "degradation_vs_exact"][cols])
print("\nSENSITIVITY: all groups incl. exploratory ones (tiny cells make this noisy)")
show(B[B.kind == "sensitivity_all_groups"][cols])
print("\nWorkload contrast (dependent − independent, SYNTHETIC masks; mean ± 95% t-interval over seeds) at the primary budget")
WC = MET("workload_contrast"); show(WC[np.isclose(WC.q, PQ) & WC.method.isin(["global_prior", "proposed", "proposed_unaware", "util_sel"])])

## 10 · Required ablations
(A1) remove fairness-aware fallback selection → `util_sel` · (A2) one global fallback → `global_prior` · (A3) disable evidence-based (barren-node) simplification · (A4) generous budgets approximating unrestricted inference.

In [ ]:
mets = ["fallback_rate", "ap", "dEOpp_supported", "worst_group_flip", "agree_exact"]
rows = []
for w in ("natural", "dependent"):
    base = R[(R.tag == "main") & np.isclose(R.q, PQ) & (R.workload == w)]
    nosimp = R[(R.tag == "ablation_no_simplification") & np.isclose(R.q, PQ) & (R.workload == w)]
    gen = R[(R.tag == "main") & np.isclose(R.q, 1.0) & (R.workload == w)]
    for label, df, meth in (("full proposed (reference)", base, "proposed"), ("A1 no fairness-aware selection", base, "util_sel"),
                            ("A2 one global fallback", base, "global_prior"), ("A3 simplification OFF (proposed)", nosimp, "proposed"),
                            ("A3 simplification OFF (global fallback)", nosimp, "global_prior"),
                            ("A4 generous budget q=1.0 (proposed)", gen, "proposed"), ("A4 generous budget q=1.0 (global fallback)", gen, "global_prior")):
        d = df[df.method == meth]
        rows.append(dict(workload=w, ablation=label, budget_ops=d.budget_ops.mean(), n_seeds=d.seed.nunique(), **{k: d[k].mean() for k in mets}))
show(pd.DataFrame(rows).set_index(["workload", "ablation"]))
print("A4: the q=1.0 budget equals the largest natural validation op count, so with natural evidence almost no query trips and the "
      "fairness gap should collapse toward the exact reference — the gap is a budget effect, not a model effect.")

## 11 · Required robustness tests
Elimination orders · network densities · discretisations · operation budgets · evidence masks · decision thresholds (capacity). All use **all 5 seeds**.

In [ ]:
robust_meth = ["exact", "global_prior", "proposed", "proposed_unaware", "util_sel", "thr_adj_post_budget"]
RR = R[R.method.isin(robust_meth)]
sel = RR[(np.isclose(RR.q, PQ)) & (RR.tag.str.startswith("robust_") | (RR.tag == "main"))]
for metric in ("dEOpp_supported", "worst_group_flip", "fallback_rate", "ap"):
    t = sel.groupby(["workload", "tag", "method"])[metric].mean().unstack("method")[[m for m in robust_meth if m in sel.method.unique()]]
    t["budget_ops"] = sel.groupby(["workload", "tag"]).budget_ops.mean()
    print(f"\n--- {metric}   (R1 robust_heuristic_*, R2 robust_structure_*, R3 robust_discretization, R6 robust_capacity_*)"); show(t.loc[["natural", "dependent"]])
print("\n--- R4 operation budgets & R5 evidence masks: full sweep (main run; workloads natural / dependent / independent)")
sw = R[(R.tag == "main") & R.method.isin(["global_prior", "proposed", "util_sel"])].groupby(["workload", "q", "budget_ops", "method"])[["fallback_gap", "worst_group_flip", "dEOpp_supported", "ap"]].mean()
show(sw.unstack("method"))

## 12 · Secondary research questions
**(1)** Which evidence patterns create the largest intermediate factors → §6 (top-pattern tables). **(2)** Does clique-tree preprocessing change the cost trade-off → §6 (calibration cost vs VE). **(3)** *When do approximation errors concentrate near decision thresholds?* Fallback-served queries only, binned by distance of the exact posterior to the decision threshold:

In [ ]:
from src.fallbacks import candidate_fallbacks
from src.pipeline import natural_budgets
fbs = candidate_fallbacks(factors, states, C["fallbacks"]); Bq = natural_budgets(factors, states, splits["val"], H, C["budget_quantiles"])
rows = []
for w in ("natural", "dependent"):
    mv_, mt_ = make_mask(splits["val"], C, bl, w, 0, 0), make_mask(splits["test"], C, bl, w, 0, 1)
    exv = predict(splits["val"], factors, states, heuristic=H, extra_mask=mv_); ext = predict(splits["test"], factors, states, heuristic=H, extra_mask=mt_)
    thr = float(np.quantile(exv.p1, 1 - C["capacity"]))
    pick = CH[(CH.tag == "main") & (CH.workload == w) & (CH.seed == 0) & np.isclose(CH.q, PQ)].iloc[0].proposed
    for name, fb in (("global_prior", "prior"), ("proposed", pick)):
        r = predict(splits["test"], factors, states, Bq[PQ], H, True, fbs[fb], mt_); m = r.used_fallback.to_numpy()
        if m.sum() >= 50:
            rows.append(error_vs_threshold(ext.p1.to_numpy()[m], r.p1.to_numpy()[m], thr).assign(workload=w, method=f"{name} ({fb})"))
if rows:
    TT = pd.concat(rows); show(TT[["workload", "method", "bin", "n", "mean_dist", "mean_tv_error", "flip_rate"]], 4)
    print("If flip_rate is highest in the 'closest' bin, approximation error matters most where the exact score sits near the cutoff; "
          "if it stays high in 'farthest' bins the fallback is simply too coarse (it ignores informative evidence).")
else:
    print("too few fallback-served queries for a binned analysis")

## 13 · Exp 5 — synthetic networks: treewidth × evidence availability (SIMULATED)
**Everything below is simulated — known structure, known availability, known exact marginals. None of it is a fact about customers.** Two groups with different evidence availability; the **control** gives both groups the same (pooled) availability, i.e. *no group-dependent computational difference*. The grid varies **treewidth** (`max_parents`) and the **availability gap** independently.

In [ ]:
S5 = MET("exp5_aggregate"); p5 = S5[S5.fallback == "prior"]
print("Main synthetic study (SIMULATED): dependent availability vs no-dependence control, by induced width")
show(p5[["max_parents", "induced_width_mean", "n_seeds", "fallback_gap_dep_mean", "fallback_gap_ind_mean", "fallback_gap_diff_mean", "fallback_gap_diff_lo", "fallback_gap_diff_hi",
         "flip_gap_diff_mean", "dEOpp_degradation_dep_mean", "dEOpp_degradation_ind_mean", "dEOpp_degradation_diff_mean", "dEOpp_degradation_diff_lo", "dEOpp_degradation_diff_hi"]])
fig("fig_synthetic_mechanism.png")

from src.synth import run_mechanism
S = C["synthetic"]; f5 = pathlib.Path(OUT) / "metrics" / "exp5b_factorial.csv"
if not f5.exists():
    print("\nrunning factorial grid (treewidth × availability gap × 5 seeds, SIMULATED) — a few minutes ...")
    rows = []
    for w in S["widths"]:
        for a1 in (0.9, 0.7, 0.5, 0.3):                          # group-1 availability; 0.9 = no group dependence (control)
            for seed in C["seeds"]:
                rows += [dict(avail_g0=0.9, avail_g1=a1, **r) for r in run_mechanism(w, a1 < 0.9, seed, (0.9, a1), n_val=1500, n_test=3000,
                                                                                    budget_q=PQ, capacity=C["capacity"], n_vars=S["n_vars"]) if r["fallback"] == "prior"]
    pd.DataFrame(rows).to_csv(f5, index=False)
F5 = pd.read_csv(f5)
gt = F5.groupby(["max_parents", "avail_g1"]).agg(induced_width=("induced_width", "mean"), n_seeds=("seed", "nunique"), fallback_gap=("fallback_gap", "mean"),
                                                   flip_gap=("flip_gap", "mean"), dEOpp_degradation=("dEOpp_degradation", "mean"))
print("\nFactorial grid (SIMULATED): availability of group 1 (group 0 fixed at 0.9; 0.9 = no group dependence = control)")
for k in ("fallback_gap", "flip_gap", "dEOpp_degradation"):
    print("\n" + k); show(gt[k].unstack("avail_g1"))
fa, axs = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, k in zip(axs, ("fallback_gap", "dEOpp_degradation")):
    for w, d in gt.reset_index().groupby("max_parents"):
        ax.plot(d.avail_g1, d[k], marker="o", label=f"max_parents={w} (width≈{d.induced_width.mean():.1f})")
    ax.set_xlabel("availability of group 1 (0.9 = control)"); ax.set_ylabel(k); ax.grid(alpha=.25); ax.invert_xaxis()
axs[0].legend(fontsize=7); fa.suptitle("SIMULATED networks — treewidth × availability gap", fontsize=9); plt.tight_layout(); plt.show()

## 14 · Wall-clock latency — measured **separately** from the deterministic op budgets
Repeated timed batches (`time.perf_counter`); hardware dependent and never used to define a budget.

In [ ]:
LAT = MET("latency"); show(LAT[["method", "ms_per_query_median", "ms_q1", "ms_q3", "batches", "queries_per_batch", "python", "machine"]], 3)

## 15 · Post-hoc diagnostics: why validation-chosen fallbacks can fail on the test period
> **EXPLORATORY / POST-HOC.** Nothing below is used to select, tune or change any method. The "oracle" column peeks at test outcomes and is a *diagnostic only* — it is not a method and must never be reported as one.

The release is date-ordered and split positionally, so the **base rate drifts** between train, validation and test. A fallback chosen on validation can therefore rank differently on test.

In [ ]:
print("Base rate by chronological split, and realised outreach rate at the validation-fixed threshold (exact inference, test):")
zn = np.load(f"{OUT}/runs/artifacts_natural.npz", allow_pickle=True)
show(pd.DataFrame({"positive_rate": {k: v.y.mean() for k, v in splits.items()},
                   "target_capacity": {k: C["capacity"] for k in splits},
                   "realised_selection_rate_exact": {"train": np.nan, "val": C["capacity"], "test": float(zn["d_exact"].mean())}}))
print("Thresholds are fixed on validation, so the realised test selection rate differs from the nominal capacity whenever the base rate / score distribution shifts.")

cands = {"global_prior": "prior", "cand_blanket_small": "blanket_small", "cand_blanket_medium": "blanket_medium", "cand_age_prior": "age_prior"}
rows = []
for w in ("natural", "dependent"):
    for meth, fbn in cands.items():
        t = R[(R.tag == "main") & np.isclose(R.q, PQ) & (R.workload == w) & (R.method == meth)]
        c = CH[(CH.tag == "main") & np.isclose(CH.q, PQ) & (CH.workload == w)]
        rows.append(dict(workload=w, candidate=fbn, uses_protected=bool(C["fallbacks"][fbn].get("uses_protected", False)),
                         VAL_ap_loss=c[f"val_{fbn}_ap_loss"].mean(), VAL_worst_flip=c[f"val_{fbn}_worst_flip"].mean(),
                         TEST_ap_loss=(R[(R.tag == "main") & np.isclose(R.q, PQ) & (R.workload == w) & (R.method == "exact")].ap.mean() - t.ap.mean()),
                         TEST_worst_flip=t.worst_group_flip.mean(), TEST_dEOpp_supported=t.dEOpp_supported.mean()))
V = pd.DataFrame(rows)
print("\nValidation statistics the rule used  vs  what each candidate actually did on test (mean over seeds):"); show(V.set_index(["workload", "candidate"]))
for w in ("natural", "dependent"):
    v = V[V.workload == w]; ok = v[v.VAL_ap_loss <= TOL]
    pick_val = ok.sort_values(["VAL_worst_flip", "VAL_ap_loss"]).candidate.iloc[0] if len(ok) else "(none feasible)"
    best = v.TEST_worst_flip.min(); oracle = v[v.TEST_worst_flip <= best + 0.01].candidate.tolist()      # ties within 0.01 count as equal
    print(f"{w}: validation picks '{pick_val}'; ORACLE (test worst-flip, ties within 0.01) = {oracle}  "
          + ("-> consistent: the validation choice transferred to the test period" if pick_val in oracle
             else "-> MISMATCH: the validation estimate did not transfer to the test period (distribution shift / small groups)"))

## 16 · Findings (generated from the numbers above), checklist and ethics
Sentences are produced from the computed results so they stay correct whatever the real data say. A negative or null result is reported as such.

In [ ]:
def verdict(r):
    if r.hi < 0: return "proposed has LOWER ΔEOpp (CI excludes 0)"
    if r.lo > 0: return "proposed has HIGHER ΔEOpp, i.e. is WORSE (CI excludes 0)"
    return "no detectable difference (CI includes 0)"
Bc = B[(B.kind == "proposed_minus_comparator")]
print("MECHANISM")
wc = WC[np.isclose(WC.q, PQ) & (WC.method == "global_prior")].iloc[0]
nat_gap = A.loc[("natural", "global_prior"), "fallback_gap"]
print(f" • Natural evidence only (REAL): at {BUD[PQ]} ops the global-fallback rate gap between supported groups is {nat_gap:.3f} "
      f"(overall fallback rate {A.loc[('natural','global_prior'),'fallback_rate']:.3f}).")
print(f" • SYNTHETIC group-dependent masking vs matched control (global fallback, mean over seeds): fallback-rate-gap difference "
      f"{wc.fallback_gap_diff_mean:+.3f} [{wc.fallback_gap_diff_lo:+.3f}, {wc.fallback_gap_diff_hi:+.3f}].")
ctl = p5.fallback_gap_ind_mean.mean(); dep = p5.fallback_gap_dep_mean.mean()
print(f" • SIMULATED networks: mean fallback-rate gap {dep:.3f} with group-dependent availability vs {ctl:.3f} in the no-dependence control "
      f"-> the disparity is {'produced by the evidence-dependent workload' if dep > ctl + 0.05 else 'NOT clearly produced by the workload in this simulation'}.")
print("\nMITIGATION (primary question: can a budget-aware fallback reduce the fairness degradation at a fixed budget?)")
for _, r in Bc[Bc.confirmatory].iterrows():
    print(f" • CONFIRMATORY, natural, {r.comparison}: {r.estimate:+.3f} [{r.lo:+.3f}, {r.hi:+.3f}], Holm p={r.p_holm:.3f} -> {verdict(r)}")
for _, r in Bc[(Bc.workload == "dependent")].iterrows():
    print(f" • exploratory, dependent (SYNTHETIC), {r.comparison}: {r.estimate:+.3f} [{r.lo:+.3f}, {r.hi:+.3f}] -> {verdict(r)}")
pk = CH[(CH.tag == "main") & (CH.workload == "dependent") & np.isclose(CH.q, PQ)].proposed.value_counts().to_dict()
print(f" • fallbacks chosen by the proposed rule (dependent, primary budget): {pk}; group-unaware variant: "
      f"{CH[(CH.tag == 'main') & (CH.workload == 'dependent') & np.isclose(CH.q, PQ)].proposed_unaware.value_counts().to_dict()}")
conf_ok = (Bc[Bc.confirmatory].hi < 0).all() and (Bc[Bc.confirmatory].p_holm < 0.05).all()
print("\nOVERALL:", "the proposed rule reduces ΔEOpp in all confirmatory comparisons." if conf_ok else
      "the mitigation claim is NOT confirmed in the pre-declared confirmatory family — report this as a (correctly qualified) negative/null finding, with the §15 diagnostics as the explanation to investigate.")
print("\nLIMITS: positional split with strong base-rate drift; no customer IDs (repeated-customer dependence uncontrolled); contacted-sample selection "
      "(historical subscriptions among contacted customers do not describe never-contacted people); synthetic masks are simulated; no causal marketing claims.")

In [ ]:
checks = []
def chk(name, ok): checks.append(dict(requirement=name, status="PASS" if bool(ok) else "FAIL"))
EJ = json.load(open(f"{OUT}/metrics/exp1_exact_agreement.json")); CJ = json.load(open(f"{OUT}/metrics/cgm.json"))
chk("Pre-contact only: no 'duration'/post-contact column in any split", not any("duration" in c.lower() for f in splits.values() for c in f.columns))
chk("Provenance recorded (SHA-256, rows, access date)", pathlib.Path(f"{OUT}/logs/data_provenance.json").exists())
chk("Dataset matches published UCI counts (41,188 x 21; 4,640 yes)", prov["matches_published_uci_counts"])
chk("Exp 1: VE (3 heuristics) = clique tree = enumeration <= 1e-6 (fitted BN, real + masked evidence)", all(v["passed"] for v in EJ.values()))
chk("Exp 1: random-network C-GM passes", CJ["passed"])
chk("VE with >=2 elimination heuristics + clique tree + sum-product calibration", {"ve_min_fill", "ve_min_weight", "clique_tree_min_fill"} <= set(X2.method))
chk("Factor-entry ops and peak factor size measured", {"ops_mean", "peak_factor_max"} <= set(X2.columns))
chk("Exp 2: cost/peak/memory across natural + synthetic workloads", X2.workload.nunique() >= 4 and "mem_peak_kb" in X2)
chk("Deterministic operation budgets (op counts, not wall-clock)", len(BUD) == len(C["budget_quantiles"]))
chk("Exp 3: fallback rate + flips by group + approximation error", {"fallback_rate", "flip"} <= set(GT.columns) and "tv_max" in R)
chk("Exp 4: proposed vs global/util/threshold-adjusted/fair-exact at matched utility", {"proposed", "global_prior", "util_sel", "thr_adj_post_budget", "fair_exact"} <= set(R.method))
chk("Group-unaware routing comparison (proposed_unaware never uses protected attr)", "proposed_unaware" in set(R.method) and not R[R.method == "proposed_unaware"].group_label_at_decision.any())
chk("Predictive baselines: logistic regression + gradient-boosted trees", {"logreg", "gbt"} <= set(PB.method))
chk("Exp 5: synthetic networks incl. no-group-dependence control", S5.shape[0] > 0 and f5.exists())
chk("Ablations A1-A4 present", {"util_sel", "global_prior"} <= set(R.method) and "ablation_no_simplification" in set(R.tag) and (np.isclose(R.q, 1.0)).any())
chk("Robustness: heuristics, densities, discretisation, budgets, masks, thresholds", {"robust_heuristic_min_weight", "robust_structure_sparse", "robust_structure_dense", "robust_discretization", "robust_capacity_0.1"} <= set(R.tag) and R.workload.nunique() == 3)
chk(">= 5 seeds for every stochastic part", len(C["seeds"]) >= 5 and R[R.workload == "dependent"].groupby("tag").seed.nunique().min() >= 5)
chk(">= 1,000 bootstrap replicates, block resampling", B.n_boot.min() >= 1000)
chk("Holm correction on the declared confirmatory family", B.loc[B.confirmatory, "p_holm"].notna().all())
chk("Small groups: denominators reported, <30 flagged exploratory", {"exploratory_tpr", "exploratory_fpr"} <= set(gd.columns))
chk("AUROC, AP, balanced accuracy, decision agreement, log loss, Brier, max & groupwise marginal error, TV", {"auroc", "ap", "balanced_acc", "agree_exact", "log_loss", "brier", "tv_max", "tv_mean"} <= set(R.columns))
chk("Wall-clock latency measured separately in repeated batches", (LAT.batches >= 3).all())
chk("Elimination traces, induced cliques, max clique size, evidence propagation shown (§4)", True)
chk("Synthetic masks / networks labelled SYNTHETIC / SIMULATED", True)
CK = pd.DataFrame(checks); display(CK)
print("ALL CHECKS PASS" if (CK.status == "PASS").all() else "Some checks FAILED - investigate before submitting (a FAIL on the published-counts check means the file is not the genuine UCI release)")
json.dump(dict(provenance=prov, checks=CK.to_dict("records"), budgets=BUD, seeds=C["seeds"]), open(f"{OUT}/summary.json", "w"), indent=1, default=str)

### Ethical questions (drafts for the report — complete in your own words)
* **Should missing evidence lead to slower or less reliable decisions?** Not by default: the budget trip rate by group (§7) shows who pays. Customers with more `unknown` fields (older/younger groups, see §1) are exactly those pushed to coarser fallbacks, so reliability should be audited by group and a group-unaware fallback preferred (§8).
* **Who bears the cost of fallback errors?** Customers wrongly *not* selected lose an offer (lost TPR); customers wrongly selected are contacted unnecessarily. §7 reports both via decision flips and TPR by group; small groups are exploratory, not "fair by lack of evidence".
* **Can historical subscription among contacted customers justify conclusions about people never contacted?** No — the data are conditional on having been contacted (selection into the sample). All results are about this audited pipeline, not about causal marketing effects or the general population.

### Before you submit
Replace every `fillin` placeholder (names, roll numbers, commit, interpretation), recompile `report/` and `slides/` from `results/` (never hand-type numbers), complete `docs/contributions.md`, extend `docs/related_work.md`, and disclose AI assistance per course policy. Both students must be able to explain every cell above.